In [ ]:
import pandas as pd
import yfinance as yf

# 1. Target the Maruti Suzuki ticker on the Indian stock exchange
ticker = yf.Ticker("MARUTI.NS")

print("Downloading Maruti Suzuki financial data...")

# 2. Extract the last 4-5 years of the three core financial statements
income_stmt = ticker.financials
balance_sheet = ticker.balance_sheet
cash_flow = ticker.cashflow

# 3. Save these statements to your computer as CSV files
income_stmt.to_csv("maruti_income_statement.csv")
balance_sheet.to_csv("maruti_balance_sheet.csv")
cash_flow.to_csv("maruti_cash_flow.csv")

print("Success! Your 3 CSV files are saved in your folder.")

Success! Your 3 CSV files are saved in your folder.


In [1]:
!pip install PyPDF2 transformers torch

In [2]:
import pandas as pd
import PyPDF2
from transformers import pipeline

# 1. Initialize FinBERT
print("Loading FinBERT...")
nlp = pipeline("sentiment-analysis", model="ProsusAI/finbert")

# 2. Hardcode the exact filenames in chronological order
file_list = [
    "q2 ft 26.pdf",     # Oct 2025
    "q3 fy 26 .pdf",    # Jan 2026
    "q4 fy 26.pdf",     # Apr 2026
    "q1 ft27.pdf"       # Jul 2026
]

all_results = []

# 3. Loop through your specific files directly
for filename in file_list:
    print(f"Processing {filename}...")

    # Extract text directly from the current directory
    with open(filename, 'rb') as pdf_file:
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        full_text = "".join([page.extract_text() for page in pdf_reader.pages])

    # Chunk into sentences
    sentences = [s for s in full_text.split('. ') if len(s.split()) > 5]

    # Run NLP on the sentences
    results = nlp(sentences)

    # Clean up the quarter names for the final presentation table
    clean_quarter_name = filename.replace('.pdf', '').strip().upper().replace('FT', 'FY')

    # Tag each sentence with its corresponding quarter
    for res, sentence in zip(results, sentences):
        res['Quarter'] = clean_quarter_name
        res['Statement'] = sentence
        all_results.append(res)

# 4. Aggregate the data into a Master Trendline
df = pd.DataFrame(all_results)
trend_summary = df.groupby(['Quarter', 'label']).size().unstack(fill_value=0)

# Reorder index to ensure chronological output
trend_summary = trend_summary.reindex(["Q2 FY 26", "Q3 FY 26", "Q4 FY 26", "Q1 FY27"])

print("\n--- Maruti Suzuki Sentiment Trend Across Quarters ---")
print(trend_summary)

Loading FinBERT...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Processing q2 ft 26.pdf...
Processing q3 fy 26 .pdf...
Processing q4 fy 26.pdf...
Processing q1 ft27.pdf...

--- Maruti Suzuki Sentiment Trend Across Quarters ---
label     negative  neutral  positive
Quarter                              
Q2 FY 26        15      173        69
Q3 FY 26        18      236        64
Q4 FY 26        17      174        95
Q1 FY27         24      168        75
